### Import libaries

In [61]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import sigmoid_kernel,linear_kernel,rbf_kernel,polynomial_kernel
from sklearn.metrics.pairwise import cosine_similarity,euclidean_distances

#### Load Dataset

In [62]:
# Load the books data set
df1=pd.read_csv("Books.csv",encoding='latin-1')  
# Load the Users data set
df2=pd.read_csv("Users.csv",encoding='latin-1')
# Load the Ratings data set
df3=pd.read_csv("Ratings.csv",encoding='latin-1')

#### Preprocessing for Books Dataset df1

In [63]:
df1.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [64]:
df1.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [65]:
df1['Book-Author']=df1['Book-Author'].fillna('Unknown')

In [66]:
df1.duplicated().sum()

np.int64(0)

In [67]:
df1.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [68]:
books=df1[['ISBN', 'Book-Title', 'Book-Author']]

In [69]:
books.head()

,ISBN,Book-Title,Book-Author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


#### Preprocessing for Users Dataset df2

In [70]:
df2.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [71]:
df2.isnull().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [72]:
# Convert 'NaN' string to actual NaN
df2['Age'] = df2['Age'].replace('NaN', np.nan)
# Fill missing age
df2['Age'] = df2['Age'].fillna(df2['Age'].median())


In [73]:
# Remove unrealistic ages
df2_new = df2[(df2['Age'] < 5) | (df2['Age'] >100)]
df2_new

,User-ID,Location,Age
219,220,"bogota, bogota, colombia",0.0
469,470,"indianapolis, indiana, usa",0.0
561,562,"adfdaf, australian capital territory, albania",0.0
612,613,"ankara, n/a, turkey",1.0
670,671,"jeddah, jeddah, saudi arabia",1.0
...,...,...,...
277075,277076,"batam, riau, indonesia",3.0
277107,277108,"quinto, ticino, switzerland",104.0
277503,277504,"san diego, california, usa",103.0
277908,277909,"phoenix, arizona, usa",2.0


In [74]:
df2_new.shape[0]

1248

In [75]:
df2['Age'].describe()

count    278858.000000
mean         33.658568
std          11.282618
min           0.000000
25%          29.000000
50%          32.000000
75%          35.000000
max         244.000000
Name: Age, dtype: float64

In [76]:
df2.isnull().sum()

User-ID     0
Location    0
Age         0
dtype: int64

In [77]:
df2['Age'].describe()

count    278858.000000
mean         33.658568
std          11.282618
min           0.000000
25%          29.000000
50%          32.000000
75%          35.000000
max         244.000000
Name: Age, dtype: float64

In [78]:
import numpy as np

# Step 1: Replace unrealistic ages with NaN
df2.loc[(df2['Age'] < 5) | (df2['Age'] > 100), 'Age'] = np.nan

# Step 2: Fill missing values with median age
df2['Age'] = df2['Age'].fillna(df2['Age'].median())

In [79]:
df2['Age'].describe()

count    278858.000000
mean         33.643385
std          10.630979
min           5.000000
25%          29.000000
50%          32.000000
75%          35.000000
max         100.000000
Name: Age, dtype: float64

In [80]:
df2[['city', 'state', 'country']] = df2['Location'].str.split(',', n=2, expand=True)

In [81]:
df2['city'] = df2['city'].str.strip()
df2['state'] = df2['state'].str.strip()
df2['country'] = df2['country'].str.strip()

In [82]:
df2 = df2.drop('Location', axis=1)

In [83]:
df2.head()

,User-ID,Age,city,state,country
0,1,32.0,nyc,new york,usa
1,2,18.0,stockton,california,usa
2,3,32.0,moscow,yukon territory,russia
3,4,17.0,porto,v.n.gaia,portugal
4,5,32.0,farnborough,hants,united kingdom


#### Preprocessing for Ratings Dataset df3

In [84]:
df3.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [85]:
df3.isnull().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

In [86]:
df3.duplicated().sum()

np.int64(0)

In [87]:
df3['Book-Rating'].value_counts()

Book-Rating
0     716109
8     103736
10     78610
7      76457
9      67541
5      50974
6      36924
4       8904
3       5996
2       2759
1       1770
Name: count, dtype: int64

In [88]:
ratings = df3[df3['Book-Rating'] != 0]
ratings

,User-ID,ISBN,Book-Rating
1,276726,0155061224,5
3,276729,052165615X,3
4,276729,0521795028,6
6,276736,3257224281,8
7,276737,0600570967,6
...,...,...,...
1149773,276704,0806917695,5
1149775,276704,1563526298,9
1149777,276709,0515107662,10
1149778,276721,0590442449,10


##### For recommendation systems, explicit ratings are easier to use.

In [89]:
ratings['Book-Rating'].value_counts()

Book-Rating
8     103736
10     78610
7      76457
9      67541
5      50974
6      36924
4       8904
3       5996
2       2759
1       1770
Name: count, dtype: int64

In [90]:
ratings.nunique().sum()

np.int64(263788)

#### Merge Datasets

In [91]:
merge1=ratings.merge(books,on='ISBN')
merge1

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author
0,276726,0155061224,5,Rites of Passage,Judith Rae
1,276729,052165615X,3,Help!: Level 1,Philip Prowse
2,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather
3,276744,038550120X,7,A Painted House,JOHN GRISHAM
4,276747,0060517794,9,Little Altars Everywhere,Rebecca Wells
...,...,...,...,...,...
383837,276704,0743211383,7,Dreamcatcher,Stephen King
383838,276704,0806917695,5,Perplexing Lateral Thinking Puzzles: Scholasti...,Paul Sloane
383839,276704,1563526298,9,Get Clark Smart : The Ultimate Guide for the S...,Clark Howard
383840,276709,0515107662,10,The Sherbrooke Bride (Bride Trilogy (Paperback)),Catherine Coulter


In [92]:
merge1.shape

(383842, 5)

In [93]:
df=merge1.merge(df2,on='User-ID',how='inner')

In [94]:
df.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Age,city,state,country
0,276726,0155061224,5,Rites of Passage,Judith Rae,32.0,seattle,washington,usa
1,276729,052165615X,3,Help!: Level 1,Philip Prowse,16.0,rijeka,n/a,croatia
2,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,16.0,rijeka,n/a,croatia
3,276744,038550120X,7,A Painted House,JOHN GRISHAM,32.0,torrance,california,usa
4,276747,0060517794,9,Little Altars Everywhere,Rebecca Wells,25.0,iowa city,iowa,usa


In [95]:
df.shape

(383842, 9)

In [96]:
df.columns

Index(['User-ID', 'ISBN', 'Book-Rating', 'Book-Title', 'Book-Author', 'Age',
       'city', 'state', 'country'],
      dtype='object')

In [97]:
df.isnull().sum()

User-ID        0
ISBN           0
Book-Rating    0
Book-Title     0
Book-Author    0
Age            0
city           0
state          0
country        0
dtype: int64

In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 383842 entries, 0 to 383841
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   User-ID      383842 non-null  int64  
 1   ISBN         383842 non-null  object 
 2   Book-Rating  383842 non-null  int64  
 3   Book-Title   383842 non-null  object 
 4   Book-Author  383842 non-null  object 
 5   Age          383842 non-null  float64
 6   city         383842 non-null  object 
 7   state        383842 non-null  object 
 8   country      383842 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 26.4+ MB


#### Filter

 We filter active users and popular items the reduce sparsity and improve recommendation quality.

In [99]:
# Step 1: Filter popular books
book_counts = df['Book-Title'].value_counts()
popular_books = book_counts[book_counts >= 50].index
df_filtered = df[df['Book-Title'].isin(popular_books)]
# Step 2: Filter active users (after book filtering)
user_counts = df_filtered['User-ID'].value_counts()
active_users = user_counts[user_counts >= 20].index
df_filtered = df_filtered[df_filtered['User-ID'].isin(active_users)]

In [100]:
# Create pivot table 
user_book_matrix = df_filtered.pivot_table(
    index='User-ID',
    columns='Book-Title',
    values='Book-Rating'
).fillna(0)

In [101]:
user_book_matrix.head()

Book-Title,1984,1st to Die: A Novel,2nd Chance,4 Blondes,84 Charing Cross Road,A Beautiful Mind: The Life of Mathematical Genius and Nobel Laureate John Nash,A Bend in the Road,A Case of Need,"A Child Called \It\"": One Child's Courage to Survive""",A Confederacy of Dunces (Evergreen Book),...,Wifey,Wild Animus,Winter Solstice,Wish You Well,Without Remorse,"Wizard and Glass (The Dark Tower, Book 4)",Wuthering Heights,Year of Wonders,Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,"\O\"" Is for Outlaw"""
User-ID,,,,,,,,,,,,,,,,,,,,,
638,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6242,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0
6251,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6543,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### cosine similarity

#### I) USER–USER SIMILARITY
#### Step 1: Compute similarity (Cosine)

In [102]:
from sklearn.metrics.pairwise import cosine_similarity
user_similarity = cosine_similarity(user_book_matrix)

In [103]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_book_matrix.index,
    columns=user_book_matrix.index
)

In [104]:
avg_rating = df.groupby('Book-Title')['Book-Rating'].mean().reset_index()
avg_rating.rename(columns={'Book-Rating':'Avg_Rating'}, inplace=True)

avg_rating.head()

,Book-Title,Avg_Rating
0,A Light in the Storm: The Civil War Diary of ...,9.000000
1,"Ask Lily (Young Women of Faith: Lily Series, ...",8.000000
2,Dark Justice,10.000000
3,Earth Prayers From around the World: 365 Pray...,7.142857
4,Final Fantasy Anthology: Official Strategy Gu...,10.000000


In [105]:
rating_count = df.groupby('Book-Title')['Book-Rating'].count().reset_index()
rating_count.rename(columns={'Book-Rating':'Rating_Count'}, inplace=True)
rating_count.head()

,Book-Title,Rating_Count
0,A Light in the Storm: The Civil War Diary of ...,1
1,"Ask Lily (Young Women of Faith: Lily Series, ...",1
2,Dark Justice,1
3,Earth Prayers From around the World: 365 Pray...,7
4,Final Fantasy Anthology: Official Strategy Gu...,2


In [106]:
author_popularity = df.groupby('Book-Author')['Book-Rating'].mean().reset_index()
author_popularity.rename(columns={'Book-Rating':'Author_Avg_Rating'}, inplace=True)
author_popularity.head()

,Book-Author,Author_Avg_Rating
0,D. Chiel,10.0
1,Mimma Balia,8.0
2,142 moms from all over the world,5.0
3,2000,7.0
4,73 Magazine Editors,6.0


#### Weighted Score:

In [107]:
book_score = avg_rating.merge(rating_count, on='Book-Title')

In [108]:
book_score['Weighted_Score'] = (
    book_score['Avg_Rating'] * book_score['Rating_Count']
)

In [109]:
book_score.head()

,Book-Title,Avg_Rating,Rating_Count,Weighted_Score
0,A Light in the Storm: The Civil War Diary of ...,9.000000,1,9.0
1,"Ask Lily (Young Women of Faith: Lily Series, ...",8.000000,1,8.0
2,Dark Justice,10.000000,1,10.0
3,Earth Prayers From around the World: 365 Pray...,7.142857,7,50.0
4,Final Fantasy Anthology: Official Strategy Gu...,10.000000,2,20.0


In [110]:
book_features = avg_rating.merge(rating_count, on='Book-Title')

In [111]:
##Adding author
book_author = df[['Book-Title','Book-Author']].drop_duplicates()
book_features = book_features.merge(book_author, on='Book-Title')

In [112]:
#Adding author popularity
book_features = book_features.merge(author_popularity, on='Book-Author')

In [113]:
book_features = book_features.merge(
    book_score[['Book-Title','Weighted_Score']],
    on='Book-Title'
)

In [114]:
book_features.head()

,Book-Title,Avg_Rating,Rating_Count,Book-Author,Author_Avg_Rating,Weighted_Score
0,A Light in the Storm: The Civil War Diary of ...,9.000000,1,Karen Hesse,7.672727,9.0
1,"Ask Lily (Young Women of Faith: Lily Series, ...",8.000000,1,Nancy N. Rue,7.529412,8.0
2,Dark Justice,10.000000,1,Jack Higgins,6.860000,10.0
3,Earth Prayers From around the World: 365 Pray...,7.142857,7,Elizabeth Roberts,7.500000,50.0
4,Final Fantasy Anthology: Official Strategy Gu...,10.000000,2,David Cassady,9.100000,20.0


In [115]:
book_features.columns

Index(['Book-Title', 'Avg_Rating', 'Rating_Count', 'Book-Author',
       'Author_Avg_Rating', 'Weighted_Score'],
      dtype='object')

In [116]:
def recommend_books(user_id, top_n=10):

    # Check if user exists
    if user_id not in user_book_matrix.index:
        return pd.DataFrame({
            'Message': ['User ID not found']
        })

    # Get target user index
    user_index = user_book_matrix.index.get_loc(user_id)

    # Get similarity scores
    similarity_scores = list(
        enumerate(user_similarity[user_index])
    )

    # Sort similar users
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    # Get similar user IDs
    similar_user_ids = [
        user_book_matrix.index[i[0]]
        for i in similarity_scores
    ]

    # Get candidate books
    candidate_books = df[
        (df['User-ID'].isin(similar_user_ids)) &
        (df['Book-Rating'] >= 7)
    ][['ISBN', 'Book-Title']]

    # Books already rated by target user
    rated_books = df[
        df['User-ID'] == user_id
    ]['Book-Title']

    # Remove already read books
    candidate_books = candidate_books[
        ~candidate_books['Book-Title'].isin(rated_books)
    ]

    # Remove duplicates
    candidate_books = candidate_books.drop_duplicates()

    # Get recommendation details
    recommendations = book_features[
        book_features['Book-Title'].isin(
            candidate_books['Book-Title']
        )
    ]

    # Sort recommendations
    recommendations = recommendations.sort_values(
        by='Weighted_Score',
        ascending=False
    )

    # Remove duplicates
    recommendations = recommendations.drop_duplicates(
        subset=['Book-Title']
    )

    return recommendations[
        ['Book-Title',
         'Avg_Rating',
         'Weighted_Score']
    ].head(top_n)

In [117]:
recommend_books(10, top_n=3)
recommend_books(999999)  # test unknown user

,Message
0,User ID not found


In [118]:
recommend_books(638, top_n=5)

,Book-Title,Avg_Rating,Weighted_Score
118460,The Red Tent (Bestselling Backlist),8.182768,3134.0
46704,Harry Potter and the Chamber of Secrets (Book 2),8.840491,2882.0
16242,Bridget Jones's Diary,7.625995,2875.0
46723,Harry Potter and the Prisoner of Azkaban (Book 3),9.043321,2505.0
135308,Where the Heart Is (Oprah's Book Club (Paperba...,8.142373,2402.0


In [123]:
import pickle

In [124]:
pickle.dump(user_book_matrix, open('matrix.pkl','wb'))
pickle.dump(user_similarity, open('similarity.pkl','wb'))
pickle.dump(df, open('df.pkl','wb'))